In [ ]:
pyfolio 下载

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import datetime as dt

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [3]:
import tushare as ts 

In [ ]:
pro = ts.pro_api('your token')

In [ ]:
#查询当前所有正常上市交易的股票列表

data = pro.query('stock_basic', exchange='', list_status='L', fields='ts_code,symbol,name,area,industry,list_date')

In [ ]:
df = pro.daily(ts_code='000001.SZ', start_date='20180701', end_date='20180718')

#多个股票
df = pro.daily(ts_code='000001.SZ,600000.SH', start_date='20180701', end_date='20180718')


In [ ]:
pro = ts.pro_api()

#查询当前所有正常上市交易的股票列表

data = pro.stock_basic(exchange='', list_status='L', fields='ts_code,symbol,name,area,industry,list_date')

In [5]:
# Set up the input variables
symbols = ['PDD', 'BIDU', 'BGNE', 'BABA', 'HTHT','JD','YY','LEGN']
start = dt.date.today() - dt.timedelta(days=365*3)
end = dt.date.today()

In [6]:
# Create an empty DataFrame
df = pd.DataFrame()
data = []

# Loop through each stock ticker, download the stock data, and add it to the DataFrame
for symbol in symbols:
    # Download the stock data
    stock_data = yf.download(symbol, fields='price', start=start, end=end)

    # Extract the adjusted closing prices and merge with the main DataFrame
    adj_close = pd.DataFrame(stock_data['Adj Close'])
    df = pd.merge(df, adj_close, right_index=True, left_index=True, how='outer')

    # Append the stock ticker to the list of tickers with available data
    data.append(symbol)

# Set the column names of the DataFrame to the tickers with available data
df.columns = data

TypeError: download() got an unexpected keyword argument 'fields'

In [ ]:
# Drop any columns that have missing data
df = df.dropna(axis='columns')

# Calculate the percentage change in stock prices over the past 3 periods
rets = df.pct_change(periods=3)

In [ ]:
# Create a scatter plot matrix of the percentage changes in stock prices
from pandas.plotting import scatter_matrix
scatter_matrix(rets, diagonal='kde', figsize=(10, 10))

In [ ]:




# Calculate the correlation matrix of the percentage changes in stock prices
corr = rets.corr()

# Create a heatmap of the correlation matrix
plt.imshow(corr, cmap='Blues', interpolation='none')
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns)
plt.yticks(range(len(corr)), corr.columns)

# Create a bar chart of the standard deviations of the percentage changes in stock prices
plt.bar(rets.columns, rets.std(), color=['red', 'blue', 'green', 'orange', 'cyan'])
plt.title("Stock Risk")
plt.xlabel("Stock Symbols")
plt.ylabel("Standard Deviations")

# Create a bar chart of the average percentage changes in stock prices
plt.bar(rets.columns, rets.mean(), color=['red', 'blue', 'green', 'orange', 'cyan'])
plt.title("Average Returns")
plt.xlabel("Stock Symbols")
plt.ylabel("Returns")

# Create a bar chart comparing the average percentage changes in stock prices to their standard deviations
ind = np.arange(5)
width = 0.35       
plt.bar(ind, rets.mean(), width, color='g', label='Average of Returns')
plt.bar(ind + width, rets.std(), width, color='r', label='Risk of Returns')
plt.ylabel('Returns Scores')
plt.xlabel('Symbols')
plt.title('Risk vs Return')
plt.xticks(ind + width / 2, ('AAPL', 'MSFT', 'AMD', 'INTC', 'NVDA'))
plt.legend(loc='best')

# Create stacked bar charts comparing the average percentage changes in stock prices to their standard deviations
ind = [x for x, _ in enumerate(symbols)]
plt.bar(ind, rets.mean(), width=0.8, label='Average of Returns', color='b')
plt.bar(ind, rets.std(), width=0.8, label='Risk of Returns', color='r', bottom=rets.mean())
plt.xticks(ind, symbols)
plt.ylabel("Returns Score")
plt.xlabel("Symbols")
plt.legend(loc="upper right")
plt.title('Risk vs Return')
plt.subplots()
plt.show()

# Create scatter plot to show expected returns vs risk
plt.scatter(rets.mean(), rets.std())
plt.xlabel('Expected returns')
plt.ylabel('Risk')
for label, x, y in zip(rets.columns, rets.mean(), rets.std()):
    plt.title('Risk vs Expected Returns')
    plt.annotate(
        label, 
        xy = (x, y), xytext = (20, -20),
        textcoords = 'offset points', ha = 'right', va = 'bottom',
        bbox = dict(boxstyle = 'round,pad=0.7', fc = 'yellow', alpha = 0.5),
        arrowprops = dict(arrowstyle = '->', connectionstyle = 'arc3,rad=0'))
plt.subplots()
plt.show()

# Display table with risk vs expected returns
d = {'Risk':rets.std(), 'Expected Returns':rets.mean()}
print('Table: Risk vs Expected Returns')
tables = pd.DataFrame(data=d)
print (tables)